# Exercise 11 — Agentic Design Patterns

*This notebook accompanies the guest lecture by **Aman Chadha** on agentic design patterns. It is based on his primer [Agentic Design Patterns](https://aman.ai/primers/agentic-design-patterns/), which covers many more patterns than we can try here.*

In week 10 you built an agent: a model that calls tools in a loop until a task is done. Many useful LLM systems are simpler than that. They combine model calls in a fixed structure that *you* design, with ordinary code in between. Such a structure is a **design pattern**: a reusable answer to a recurring problem, like "the task is too complex for one prompt" or "the model's first answer is often wrong". You will implement five of them in plain Python, each in a few lines:

| Pattern | Problem it solves |
|---------|-------------------|
| **Prompt chaining** | A task is too complex for one prompt → split it into steps, and check each step in code |
| **Routing** | Different inputs need different handling → classify the input first, then send it to a specialised prompt |
| **Parallelization** | Independent subtasks, or unreliable answers → run calls at the same time, and vote |
| **Reflection** | The first answer is often wrong → critique it and revise it in a loop |
| **Orchestrator–workers** | The subtasks are not known in advance → let one model plan them, others do them, and combine the results |

Frameworks such as LangChain or LangGraph, which the primer uses, offer these patterns ready-made. Writing them yourself shows that there is no magic: each one is a few model calls and some control flow.

## What you will do
1. **Setup**: a model, a cache, and a usage counter.
2. **Prompt chaining**: extract, check, then write.
3. **Routing**: send each request to the right specialist.
4. **Parallelization**: concurrent calls, and majority voting.
5. **Reflection**: improve a draft with feedback from a checker.
6. **Orchestrator–workers**: plan, delegate, combine.
7. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**2.1**, **2.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell. Run it and make sure it passes before you move on. Model replies vary, so a few checks only test the structure of your code, not the exact output.

> **Compute**: with the default Groq backend, no GPU is needed and the notebook uses roughly 25,000 tokens of the free daily budget. With the local backend, use a GPU runtime (see Part 1).

In [ ]:
import hashlib
import json
import os
import re
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass

## 1. Setup

The notebook works with two **backends**; choose one in the next cell:

| `BACKEND` | Model | What you need |
|-----------|-------|---------------|
| `"groq"` (default) | `qwen/qwen3.8-27b` on the Groq API, as in weeks 7 and 10 | Your free Groq API key. No GPU. |
| `"local"` | `Qwen/Qwen2.5-1.5B-Instruct`, run on your machine with `transformers` | No account, but a GPU is strongly recommended: use a free **Colab** or **Kaggle** GPU runtime (in Colab: *Runtime → Change runtime type → T4 GPU*). On a laptop CPU every reply takes many seconds. |

The local model is about 18 times smaller than the one on Groq, so expect more mistakes. That is instructive too: compare your results with a classmate who uses the other backend.

`chat(messages)` sends a conversation (a list of `{"role": ..., "content": ...}` messages) to the model and returns the reply as a string. Replies are cached in a JSON file, so re-running a cell costs nothing. `usage` counts the calls and tokens, so you can compare what each pattern costs.

> **Keep your key secret.** Never write it into a code cell. If you set the environment variable `GROQ_API_KEY` before starting Jupyter, the cell does not ask at all.

In [ ]:
BACKEND = "groq"  # "groq" or "local"

if BACKEND == "groq":
    from groq import Groq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")
    client = Groq(max_retries=8)  # on a rate-limit error it waits and tries again (up to 8 times)
    # If this model is no longer available, pick another one from https://console.groq.com/docs/models
    MODEL = "qwen/qwen3.8-27b"
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device, "" if device == "cuda" else "(this will be slow; consider a Colab GPU runtime)")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    local_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto").to(device).eval()

CACHE_PATH = "llm_cache_patterns.json"
try:
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}
cache_lock = threading.Lock()
usage = {"calls": 0, "tokens": 0}


def _generate(messages, temperature, max_tokens, sample):
    # Returns the reply text and the number of tokens used
    if BACKEND == "groq":
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
            reasoning_effort="none",
        )
        return response.choices[0].message.content or "", response.usage.total_tokens
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    torch.manual_seed(sample)
    sampling = {"do_sample": True, "temperature": temperature} if temperature > 0 else {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
    with torch.no_grad():
        output = local_model.generate(**inputs, max_new_tokens=max_tokens, pad_token_id=tokenizer.eos_token_id, **sampling)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True), output.shape[1]


def chat(messages, temperature=0.0, max_tokens=512, sample=0):
    # Send a conversation to the model and return the reply (a string).
    # With temperature > 0, different values of `sample` give different (independently cached) samples.
    key = hashlib.sha256(json.dumps([BACKEND, MODEL, messages, temperature, max_tokens, sample]).encode()).hexdigest()
    if key not in cache:
        reply, tokens = _generate(messages, temperature, max_tokens, sample)
        with cache_lock:
            cache[key] = {"reply": reply, "tokens": tokens}
            with open(CACHE_PATH, "w", encoding="utf-8") as f:
                json.dump(cache, f)
    with cache_lock:
        usage["calls"] += 1
        usage["tokens"] += cache[key]["tokens"]
    return cache[key]["reply"]


print(chat([{"role": "user", "content": "Say hello to the NLP students in Aarhus in five words."}]))

Two helpers we use throughout: `parse_json` finds the first JSON object or list in a reply (models like to wrap JSON in text or in ```` ```json ```` fences) and returns `None` if there is none, and `show_usage` prints how many calls and tokens a piece of code used.

In [ ]:
def parse_json(text):
    # Return the first JSON object or list in `text`, or None
    decoder = json.JSONDecoder()
    for i, char in enumerate(text):
        if char in "{[":
            try:
                return decoder.raw_decode(text[i:])[0]
            except json.JSONDecodeError:
                continue
    return None


def show_usage(before, label=""):
    print(f"{label}{usage['calls'] - before['calls']} calls, {usage['tokens'] - before['tokens']} tokens")


print(parse_json('Sure! ```json\n{"a": 1, "b": [2, 3]}\n```'))
print(parse_json("no JSON here"))

## 2. Prompt chaining

**Prompt chaining** splits a task into a sequence of steps. Each step is a model call with one clear job, and its output is the input of the next step. Between the steps, ordinary code can check the intermediate result. Such a check is called a **gate**: if it fails, the chain stops (or takes another path) instead of passing on garbage.

Our task: answer customer reviews. In one prompt, the model would have to understand the review, decide whether a reply is needed, and write it. We split this into:

1. **Extract** the product, the problem and the sentiment as JSON (a model call).
2. **Gate**: check the JSON in code. If there is no problem, send a fixed thank-you note without calling the model again.
3. **Write** a reply that addresses the problem (a model call).

In [ ]:
REVIEWS = [
    "I ordered the TrailMax backpack two weeks ago. The zipper broke on the first hike and customer service hasn't answered my emails. Really disappointed.",
    "The AeroPress coffee maker works great, it's easy to clean and the coffee tastes amazing. Would buy again!",
    "My SoundWave headphones stopped charging after a month. The sound was nice while it lasted, but I want a replacement.",
]

**2.1 Implement `extract_info(review)`.** Ask the model to return a JSON object with the keys `"product"` (the product name), `"problem"` (a short description of the problem, or `null` if there is none) and `"sentiment"` (`"positive"`, `"negative"` or `"mixed"`). Use `parse_json` on the reply and return the result (a dict, or `None` if the reply contains no JSON).

In [ ]:
def extract_info(review):
    messages = ...  # TODO
    return parse_json(chat(messages))

In [ ]:
# ✅ Check extract_info
infos = [extract_info(review) for review in REVIEWS]
for info in infos:
    print(info)
    assert isinstance(info, dict), "extract_info should return a dict"
    assert {"product", "problem", "sentiment"} <= set(info), "the dict needs the keys product, problem and sentiment"
    assert info["sentiment"] in {"positive", "negative", "mixed"}, f"unexpected sentiment {info['sentiment']!r}"
assert "trailmax" in infos[0]["product"].lower() and infos[0]["problem"], "review 1 is about the TrailMax backpack and has a problem"
assert infos[1]["problem"] in (None, "", "null", "none"), "review 2 has no problem"
print("Looks good ✅")

**2.2 Implement `respond(review)`**, the full chain:
1. Call `extract_info`.
2. **Gate**: if the result is `None` or misses one of the three keys, raise a `ValueError`. If there is no problem (`None`, or an empty string), return `f"Thank you for your kind words about the {product}!"` **without** another model call.
3. Otherwise, ask the model to write a short, polite reply (at most 80 words) that names the product, apologises for the specific problem and offers a replacement. Pass it the extracted fields, not the raw review.

In [ ]:
def respond(review):
    info = extract_info(review)
    # TODO: the gate
    # TODO: the second model call
    ...

In [ ]:
# ✅ Check respond
before = dict(usage)
for review in REVIEWS:
    print(respond(review), "\n")
show_usage(before, "the chain used ")
before = dict(usage)
reply = respond(REVIEWS[1])
assert reply.startswith("Thank you") and usage["calls"] - before["calls"] == 1, "a review without a problem should take only one model call"
assert "trailmax" in respond(REVIEWS[0]).lower(), "the reply should name the product"
print("Looks good ✅")

**2.3 Why chain?** The chain uses more calls than a single prompt like *"Read this review and, if needed, write a reply"*. Name two advantages of the chain and one disadvantage.

*Your answer here*

## 3. Routing

**Routing** classifies an input first and then sends it to the path that suits it best: a specialised prompt, a tool, a cheaper or stronger model, or a different agent. The classifier can be a model call, a small trained classifier, or even keyword rules.

Our customer service desk has three specialists, each with its own system prompt:

In [ ]:
ROUTES = {
    "billing": "You are a billing specialist. Explain charges, refunds and payment methods clearly. Never ask for full card numbers.",
    "technical": "You are a technical support engineer. Ask for the device and app version if needed, and give numbered troubleshooting steps.",
    "general": "You are a friendly receptionist. Answer general questions about the company briefly.",
}

TEST_QUERIES = [
    ("I was charged twice for my subscription this month.", "billing"),
    ("The app crashes every time I open the settings page.", "technical"),
    ("How do I get a refund for my last order?", "billing"),
    ("My password reset email never arrives.", "technical"),
    ("Do you have an office in Aarhus?", "general"),
    ("Can I change the credit card on my account?", "billing"),
    ("The website shows error 500 when I upload a photo.", "technical"),
    ("What are your opening hours during Christmas?", "general"),
]

**3.1 Implement `classify(query)`.** Ask the model which of the routes fits the query, and to reply with one word. Models do not always follow the format, so parse robustly: return the first route name that appears in the reply (lowercased), and `"general"` as a fallback.

In [ ]:
def classify(query):
    ...  # TODO

In [ ]:
# ✅ Check classify
correct = 0
for query, expected in TEST_QUERIES:
    label = classify(query)
    assert label in ROUTES, f"classify should return one of {list(ROUTES)}, not {label!r}"
    correct += label == expected
    print(f"{label:10} {'✓' if label == expected else '✗ expected ' + expected:20} {query}")
assert correct >= 6, "fewer than 6 of 8 correct: improve the prompt"
print("Looks good ✅")

**3.2 Implement `handle(query)`**: classify the query, then answer it with the system prompt of that route. Return the route and the reply.

In [ ]:
def handle(query):
    route = ...  # TODO
    reply = ...  # TODO
    return route, reply

In [ ]:
for query in ["I think I paid for the premium plan but I still see ads.", "The export button does nothing on my phone."]:
    route, reply = handle(query)
    print(f"[{route}] {query}\n{reply}\n")

**3.3 Routing to models.** Routing is also used to save money: easy requests go to a small, cheap model and hard ones to a large model. What goes wrong when the router makes a mistake in each direction? How could you make a router safer?

*Your answer here*

## 4. Parallelization

When subtasks do not depend on each other, there is no reason to wait for one before starting the next. **Parallelization** comes in two flavours:

- **Sectioning**: split a task into independent parts and run them at the same time (e.g. check a text for grammar, tone and factual errors in three parallel calls).
- **Voting**: run the *same* task several times and combine the answers, e.g. by majority vote. This is the idea of **self-consistency** ([Wang et al., 2023](https://arxiv.org/abs/2203.11171)).

`parallel_map` runs a function on a list of inputs in several threads. With the Groq backend, the requests run concurrently; a local model handles one request at a time, so there we run them one after the other.

In [ ]:
def parallel_map(function, items, workers=4):
    if BACKEND != "groq":
        workers = 1
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(function, items))


def translate(language, sample):
    prompt = f"Translate into {language}, reply with the translation only: 'The library opens at nine.'"
    return chat([{"role": "user", "content": prompt}], sample=sample)  # a different `sample` avoids the cache


languages = ["Danish", "German", "Spanish", "Greek"]
start = time.time()
print([translate(language, sample=1) for language in languages])
print(f"one after the other: {time.time() - start:.1f} s")
start = time.time()
print(parallel_map(lambda language: translate(language, sample=2), languages))
print(f"in parallel:         {time.time() - start:.1f} s")
# (When you re-run the cell, the replies come from the cache and both take no time.)

For voting, we need *different* answers, so we sample with `temperature=1.0` and a different `sample` number for each call. Our question is a classic that tempts a quick, wrong answer:

In [ ]:
QUESTION = (
    "A bat and a ball cost 1.10 euros in total. The bat costs 1.00 euro more than the ball. "
    "How many cents does the ball cost? Think briefly, then finish with 'Answer: <number>'."
)


def extract_answer(reply):
    # The number after the last "Answer:", or None
    matches = re.findall(r"Answer:\D{0,15}?(-?\d+(?:\.\d+)?)", reply)
    return float(matches[-1]) if matches else None


print(extract_answer("The ball costs 5 cents. Answer: 5"), extract_answer("**Answer:** $\\boxed{5}$"))

**4.1 Implement `vote(question, n=7)`**: sample `n` replies in parallel (use `parallel_map`, with `temperature=1.0` and `sample=i` for the `i`-th call), extract the answers, and return the most common answer among those that are not `None`, together with the list of all answers.

In [ ]:
def vote(question, n=7):
    ...  # TODO

In [ ]:
# ✅ Check vote
before = dict(usage)
majority, answers = vote(QUESTION)
print("answers:", answers, " majority:", majority)
show_usage(before)
assert len(answers) == 7, "sample n answers"
valid = [a for a in answers if a is not None]
assert majority == Counter(valid).most_common(1)[0][0], "return the most common valid answer"
print("Looks good ✅")

**4.2 When does voting help?** Look at the answers. Under which condition does a majority vote give a better answer than a single sample? What does it cost, and for which kinds of task can you *not* use a simple majority vote?

*Your answer here*

## 5. Reflection

In **reflection** (also called *evaluator–optimizer*, or Self-Refine, [Madaan et al., 2023](https://arxiv.org/abs/2303.17651)), a draft is critiqued, and the critique is fed back to the model to produce a better version, in a loop until the critic is satisfied or a round limit is reached. The critic can be the model itself, another model, a human, or **code**, such as unit tests, a compiler or a rule checker. When such a checker exists, it is the most reliable critic, and we use one here.

The task asks for something language models are notoriously bad at, because they see tokens, not words: a sentence with an exact number of words.

**5.1 Implement `check_sentence(text, n_words, last_word, must_contain)`**, which returns a list of problems (strings), empty if there are none. A word is anything between spaces (`text.split()`). The rules:
- exactly `n_words` words; the message should say how many there are, e.g. `"The sentence has 14 words, but it must have exactly 12."`
- no commas
- it contains the word `must_contain`
- its last word, without punctuation (`.!?"'`), is `last_word` (ignoring case)

In [ ]:
def check_sentence(text, n_words, last_word, must_contain):
    problems = []
    # TODO
    return problems

In [ ]:
# ✅ Check check_sentence
assert check_sentence("Grey clouds drift over Aarhus as the harbour waits for rain.", 11, "rain", "Aarhus") == []
p = check_sentence("In Aarhus, it rains.", 12, "rain", "Aarhus")
assert len(p) == 3 and "4 words" in p[0], "three problems: the length (4 words), the comma, the last word"
assert check_sentence("Leaves fall in Copenhagen.", 4, "rain", "Aarhus") != []
print("Looks good ✅")

**5.2 Implement `reflect(task, checker, max_rounds=4)`.** Ask the model for a first draft (`task` plus *"Reply with the sentence only."*). Then, as long as `checker(draft)` finds problems and there are rounds left, continue the **same conversation**: append the draft as an assistant message and a user message that lists the problems and asks for a corrected sentence. Return the final draft and the list of all drafts.

In [ ]:
def reflect(task, checker, max_rounds=4):
    messages = [{"role": "user", "content": task + " Reply with the sentence only."}]
    # TODO
    ...

In [ ]:
# ✅ Check reflect
final, drafts = reflect("Write a sentence about rain in Aarhus.", lambda text: [])
assert len(drafts) == 1 and final == drafts[0], "if the checker finds no problems, stop after the first draft"
final, drafts = reflect("Write a sentence about rain in Aarhus.", lambda text: ["Always wrong."], max_rounds=2)
assert len(drafts) == 3, "with max_rounds=2 there are at most 1 + 2 drafts"
print("Looks good ✅")

In [ ]:
TASKS = [
    (12, "rain", "Aarhus", "about autumn in Aarhus"),
    (9, "snow", "Copenhagen", "about winter in Copenhagen"),
    (15, "bicycle", "student", "about a student's first week at university"),
]
before = dict(usage)
for n_words, last_word, must_contain, topic in TASKS:
    task = (f"Write one sentence {topic}. It must have exactly {n_words} words, contain no commas, "
            f"contain the word '{must_contain}' and end with the word '{last_word}'.")
    checker = lambda text: check_sentence(text, n_words, last_word, must_contain)
    final, drafts = reflect(task, checker)
    for i, draft in enumerate(drafts):
        print(f"round {i}: {'✓' if not checker(draft) else '✗'} ({len(draft.split())} words) {draft}")
    print()
show_usage(before)

**5.3 Who should critique?** How often was the first draft correct, and how often did the loop fix it? Now imagine you replace `check_sentence` by a second model call: *"Does this sentence have exactly 12 words? List any problems."* What would change? For which tasks is there no code checker, and what can you do then?

*Your answer here*

## 6. Orchestrator–workers

In prompt chaining, *you* decide the steps. Sometimes the steps depend on the input, and you cannot fix them in advance. In the **orchestrator–workers** pattern, a model (the orchestrator) **plans** the subtasks, worker calls carry them out (in parallel, since they are independent), and a final call **synthesises** the results. This is a simple form of a multi-agent system with a *supervisor* topology.

**6.1 Implement `plan(goal, max_steps=4)`.** Ask the model to split the goal into at most `max_steps` independent subtasks and to return them as a JSON list of strings. Parse the reply with `parse_json`. If the result is not a non-empty list of strings, fall back to `[goal]` (do the whole task in one step). Return at most `max_steps` subtasks.

In [ ]:
def plan(goal, max_steps=4):
    ...  # TODO

In [ ]:
# ✅ Check plan
GOAL = "Write a short welcome guide for an international student who arrives in Aarhus in January."
subtasks = plan(GOAL)
print(json.dumps(subtasks, indent=2))
assert isinstance(subtasks, list) and 1 <= len(subtasks) <= 4 and all(isinstance(s, str) for s in subtasks)
print("Looks good ✅")

**6.2 Implement `orchestrate(goal)`**: plan the subtasks, let one worker call do each subtask in parallel (each worker sees the overall goal and its own subtask, and writes at most 120 words), and finally ask the model to combine the workers' results into one coherent text for the goal. Return the final text and the subtasks.

In [ ]:
def orchestrate(goal):
    subtasks = plan(goal)
    # TODO: the workers, in parallel
    # TODO: the synthesis
    ...

In [ ]:
before = dict(usage)
final, subtasks = orchestrate(GOAL)
show_usage(before, "orchestrator-workers: ")
print(final)

before = dict(usage)
single = chat([{"role": "user", "content": GOAL}], max_tokens=800)
show_usage(before, "\nsingle call: ")

**6.3 Was it worth it?** Compare the two texts and their cost. The primer recommends to *default to a single agent and escalate deliberately*. When is the orchestrator–workers pattern (or a multi-agent system in general) worth its extra cost, and when is a single call or a single agent better?

*Your answer here*

### Going further
The primer covers many more patterns that build on these: **planning** and **tool use** (week 10), **memory** for long-running agents, **guardrails** that check inputs and outputs, **human-in-the-loop** approval for consequential actions, **exception handling**, and **evaluation** of agents. Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) describes the five patterns of this notebook with diagrams, and recommends the same principle: start with the simplest solution, and add structure only when it measurably helps.

## 7. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 7.1. Which pattern?

Incoming emails must each be answered by one of three teams with very different instructions (sales, legal, support). Which pattern fits best?

A. Parallelization with majority voting<br>
B. Routing<br>
C. Reflection<br>
D. Orchestrator–workers<br>

**Answer:** 

---

### 7.2. Gates

What is a gate in a prompt chain?

A. A special token that ends the model's reply<br>
B. A check in code between two steps that stops or redirects the chain if an intermediate result is invalid<br>
C. A model that decides which tool to call<br>
D. The rate limit of the API<br>

**Answer:** 

---

### 7.3. Voting

You use majority voting over 9 samples. For which task does it help *least*?

A. A multiple-choice question where the model is right 60% of the time and wrong answers are spread over the other options<br>
B. An arithmetic word problem with a single numeric answer<br>
C. Writing a persuasive cover letter<br>
D. Classifying a review as positive or negative<br>

**Answer:** 

---

### 7.4. Reflection

Which critic gives the most reliable feedback in a reflection loop, when it is available?

A. The same model, asked whether its answer is correct<br>
B. A larger model, asked to rate the answer from 1 to 10<br>
C. An external verifier, such as unit tests, a compiler or a proof checker<br>
D. No critic: more rounds always help<br>

**Answer:** 

---

### 7.5. Single or multi-agent

According to the primer, what is a good default when designing an LLM system?

A. Always use as many agents as there are subtasks<br>
B. Start with a single agent (or a simple workflow) and escalate to multiple agents deliberately, when the task needs it<br>
C. Always use a framework such as LangGraph<br>
D. Avoid tools, because they make agents unreliable<br>

**Answer:** 